# Guardian Pay on IBM LinuxONE

Every cell here runs on the LinuxONE server itself. It:

1. checks the machine and its on-chip AI accelerator,
2. trains the three models on LinuxONE (coercion, PaySim transaction patterns, scam text),
3. scores payments in real time and measures how long each decision takes,
4. runs IBM Granite inside LinuxONE (llama.cpp with zDNN) for the Saathi assistant and the message checker.

Open this notebook from `shared/guardian-pay/notebooks/` in Jupyter Lab and run the cells from top to bottom.

In [ ]:
import os, sys, platform, time, glob, json, subprocess
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)
print('Project folder :', ROOT)
print('Architecture   :', platform.machine())
print('Python         :', platform.python_version())
try:
    cpu = open('/proc/cpuinfo').read()
    machine = [l for l in cpu.splitlines() if l.startswith('processor 0') or 'machine' in l][:2]
    print('CPU            :', ' | '.join(machine) or 'see /proc/cpuinfo')
    print('NNPA facility  :', 'yes' if 'nnpa' in cpu.lower() else 'not reported by this VM')
except Exception as e:
    print('CPU info unavailable:', e)
import sklearn, numpy
print('scikit-learn   :', sklearn.__version__, '| numpy', numpy.__version__)
zdnn = glob.glob('/opt/venv/lib*/python3*/site-packages/llama_cpp/lib/libggml-zdnn*.so')
print('llama.cpp zDNN :', zdnn[0] if zdnn else 'not found')

## 1. Coercion model
One million payments from our generator (eight personas, including hard cases such as a real family emergency and scams whose receiving account still looks clean). Trained and tested here on LinuxONE.

In [ ]:
t = time.time()
!{sys.executable} -m guardian.train --rows 1000000
print(f'Trained on {platform.machine()} in {time.time()-t:.0f}s')

## 2. Transaction-pattern model on PaySim
PaySim is the public mobile-money dataset (6.3 million transactions, calibrated on a real operator's logs). Only facts known before money moves are used. Trained on earlier days, tested on later days.

The first run downloads the 490 MB CSV.

In [ ]:
csv = 'data/paysim/paysim.csv'
if not os.path.exists(csv):
    os.makedirs('data/paysim', exist_ok=True)
    !curl -sSL -o {csv} https://huggingface.co/datasets/theman10/paysim/resolve/main/paysim.csv
t = time.time()
!{sys.executable} -m guardian.train_paysim --csv {csv}
print(f'Trained on {platform.machine()} in {time.time()-t:.0f}s')

## 3. Scam and pressure text model
Reads whole sentences (word pairs and character pieces, so Hinglish spellings and Devanagari work). Four public datasets: UCI SMS Spam Collection, Indian cyber-scam phone calls in Hinglish, scam and ham SMS in 14 Indian languages, and a Hinglish scam text set.

The last block tests each dataset after training on the other three, to show how well it generalises.

In [ ]:
t = time.time()
!{sys.executable} -m guardian.text_model
print(f'Trained on {platform.machine()} in {time.time()-t:.0f}s')

## 4. Real-time scoring
Score 20,000 payments one by one, the way they arrive, and measure the decision time of each.

In [ ]:
import numpy as np
from guardian.engine import Engine
from guardian.generator import sample_one
eng = Engine(); rng = np.random.default_rng(5)
personas = ['everyday']*70 + ['big_legit']*10 + ['family_emergency']*5 + ['anxious_senior']*5 + ['digital_arrest']*5 + ['relative_clone']*3 + ['task_scam']*2
lat, decisions = [], {}
for i in range(20000):
    p = personas[i % len(personas)]
    out = eng.decide(sample_one(rng, p))
    lat.append(out['latency_us']); decisions.setdefault(p, []).append(out['decision'])
lat = np.array(lat)
print(f'median {np.median(lat):.0f} µs | p99 {np.percentile(lat, 99):.0f} µs | max {lat.max()} µs per decision')
for p, d in decisions.items():
    print(f'{p:18s} paused or blocked: {100*np.mean([x in ("pause","block") for x in d]):5.1f}%')

## 5. A digital-arrest payment, end to end

In [ ]:
feats = dict(app_switches=3, pasted_account=1, confirm_hesitation_s=24, amount_edits=1, retries_after_limit=1,
             session_duration_s=190, amount_ratio=203, balance_drain=0.77, new_payee=1, payee_history=0,
             fd_broken_24h=1, night_hour=0, sender_age=66, recv_account_age_days=640, recv_inbound_24h=0,
             recv_distinct_senders_24h=0, recv_senior_share=0, recv_forward_ratio=0)
r = eng.decide(feats)
print('Decision:', r['decision'], '| score', r['score'], '|', r['latency_us'], 'µs')
for x in r['reasons']: print(' -', x['en'], '/', x['hi'])
print('PaySim pattern score:', eng.paysim_score(650000, 840000, 0, 11))

## 6. IBM Granite inside LinuxONE
The Granite model that ships in IBM's image, served by llama.cpp built with zDNN, runs the Saathi conversation and reads a scam message. Nothing leaves the machine.

In [ ]:
os.environ['GUARDIAN_PREFER_LOCAL'] = '1'
from guardian.llm import AI, find_granite_gguf
from guardian import ai_features
from guardian.text_model import load_default
print('Granite model:', find_granite_gguf())
ai = AI(); tm = load_default()
print('AI order:', ai.status())
txn = {'id':'NB1','amount':650000,'payee':{'name':'R K Enterprises','account':'50100447122906'},
       'sender':{'name':'Suresh Sharma','age':66,'city':'Lucknow'},'reasons':r['reasons'],'rules':[],'features':feats}
for said in [None, 'ek officer video call pe hai, bol rahe hain parcel mein drugs mile hain']:
    t = time.time(); out = ai_features.saathi_turn(ai, txn, said, tm)
    print(f"\nCustomer: {said}\nSaathi ({out['provider']}, {time.time()-t:.1f}s): {out['reply']}\nRisk: {out['risk']}")

In [ ]:
msg = ('Main CBI officer bol raha hoon. Aapke naam ke parcel mein drugs mile hain. Verification ke liye '
       '5010 0447 1229 06 mein 4,50,000 bhejiye, kisi ko mat batana warna arrest hoga.')
t = time.time(); res = ai_features.check_message(ai, msg, None, tm)
print(f"{res['verdict']} ({res['scam_type']}) via {res['provider']} in {time.time()-t:.1f}s")
print('Red flags:', res['red_flags'])
print('Text model:', res.get('ml'))
print(res['advice_hi'])

## 7. Results

In [ ]:
for name, path in [('Coercion model', 'model/guardian_model.json'), ('PaySim model', 'model/paysim_model.json'),
                   ('Text model', 'model/text_model.json')]:
    if os.path.exists(path):
        m = json.load(open(path))['metrics']
        keep = {k: m[k] for k in ('rows','rows_used','auc','avg_precision','at_70','at_80','scam_holdout','scam_unseen_source') if k in m}
        print(name); print(json.dumps(keep, indent=1, ensure_ascii=False)[:1500]); print()